# Building models with torch.nn: nn.Module, layers, and parameters

_PyTorch (a complete course)_

**Subclass nn.Module, declare layers in __init__, wire them in forward, and PyTorch tracks every weight for you.**

An nn.Module bundles two things: the state (its weight tensors, called parameters) and
       the computation (how to turn an input into an output). You define both in one class.

       The pattern never changes: declare the layers in __init__, then describe the data
       flow in forward. PyTorch does the rest &mdash; it finds every parameter, tracks gradients,
       and lets you save, load, and move the whole model with one call.

---

This notebook is a practice scaffold for the **Building models with torch.nn: nn.Module, layers, and parameters** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)                      # reproducible weights and input

# ============================================================
# 1. AN MLP BY SUBCLASSING nn.Module
#    Layers live in __init__ (so they are REGISTERED);
#    the data flow lives in forward.
# ============================================================
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10):
        super().__init__()               # MUST be first: sets up registration
        self.fc1 = nn.Linear(in_dim, hidden)   # registered weight + bias
        self.relu = nn.ReLU()                  # parameter-free activation
        self.fc2 = nn.Linear(hidden, out_dim)  # registered weight + bias

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x                          # raw logits (no softmax here)

model = MLP()
print(model)                              # prints the module tree

# ------------------------------------------------------------
# Parameters are auto-registered: each is an autograd tensor.
# ------------------------------------------------------------
n_params = sum(p.numel() for p in model.parameters())
print("trainable parameters:", n_params)          # -> 235146
print("fc1.weight requires_grad:",
      model.fc1.weight.requires_grad)             # -> True
print("state_dict keys:", list(model.state_dict().keys()))
#   -> ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias']

# ------------------------------------------------------------
# A forward pass on a random batch of 8 examples.
# CALL THE MODEL, not .forward(): model(x) runs hooks + forward.
# ------------------------------------------------------------
x = torch.randn(8, 784)                   # batch of 8, 784 features each
logits = model(x)                         # NOT model.forward(x)
print("output shape:", logits.shape)      # -> torch.Size([8, 10])

# ============================================================
# 2. THE SAME MLP AS nn.Sequential (quick stack, no class)
#    Use this when the model is just a straight chain.
# ============================================================
seq = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10),
)
seq_params = sum(p.numel() for p in seq.parameters())
print("sequential parameters:", seq_params)       # -> 235146 (identical)
print("sequential output:", seq(x).shape)         # -> torch.Size([8, 10])

# ------------------------------------------------------------
# Weight initialization: override the defaults if you want.
# (nn.Linear already inits sensibly; this shows how to control it.)
# ------------------------------------------------------------
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        nn.init.zeros_(m.bias)

model.apply(init_weights)                 # walks every submodule
print("after re-init, fc1.bias is all zeros:",
      bool(torch.all(model.fc1.bias == 0)))        # -> True

## Visualize the data & results

_Where do an MLP's parameters actually live? Per-layer trainable-parameter count for the MLP Linear(784->256) -> ReLU -> Linear(256->128) -> ReLU -> Linear(128->10), computed with the formula in*out + out._

In [ ]:
import numpy as np

# An MLP described as a list of (in_features, out_features) for each Linear,
# with parameter-free ReLU activations in between.
linears = [(784, 256), (256, 128), (128, 10)]

labels, values = [], []
for (in_f, out_f) in linears:
    params = in_f * out_f + out_f        # weight matrix (in*out) + bias (out)
    labels.append(f"Linear {in_f}->{out_f}")
    values.append(params)
    labels.append("ReLU")                # activation between Linear layers...
    values.append(0)                     # ...has zero parameters
labels, values = labels[:-1], values[:-1]   # drop trailing ReLU after last Linear

for lab, v in zip(labels, values):
    print(f"{lab:18s} {v:>8d}")
print("total parameters:", sum(values))   # -> 235146

# (matches torch:  sum(p.numel() for p in model.parameters()))
import matplotlib.pyplot as plt
plt.bar(labels, values, color=["#4ea1ff","#8b949e","#4ea1ff","#8b949e","#7ee787"])
plt.ylabel("parameter count")
plt.title("Trainable parameters per layer of a 784-256-128-10 MLP")
plt.xticks(rotation=20)
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate's model trains but the hidden layer's weights never change &mdash; the loss barely moves. Their code creates the hidden layer like this: def forward(self, x): h = nn.Linear(8, 8)(x); return self.out(h). What is wrong, and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Spot that the hidden nn.Linear is built inside forward, not in __init__. — _Only layers assigned to self in __init__ get registered; a layer created inside forward is a throwaway local._
- Realize the layer is recreated with fresh random weights on every forward pass and is absent from model.parameters(). — _If it is not in parameters(), the optimizer never receives it, so its weights are never updated &mdash; and it is reset each call anyway._
- Move it to __init__: self.hidden = nn.Linear(8, 8), then use self.hidden(x) in forward. — _Now the layer is registered once, its weights persist, and the optimizer trains them._

**Answer:** The hidden layer is constructed inside forward, so it is never registered: it is missing from model.parameters() and the optimizer never updates it &mdash; worse, it is rebuilt with new random weights every call. Define layers in __init__: self.hidden = nn.Linear(8, 8), and call self.hidden(x) in forward. Then it is registered once and trains normally.

</details>

**Problem 2.** You build an MLP and run it, but get RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu. You already did x = x.to("cuda"). What did you forget, and why does it matter?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the input x is on the GPU (cuda:0) but something is still on the CPU. — _The error fires whenever a layer's weights and the data it multiplies live on different devices._
- Check whether the model itself was moved. — _Moving the input does not move the model; the nn.Linear weights stay on the CPU until you move the module._
- Add model.to("cuda") (or device) before the forward pass. — _model.to(device) walks the module tree and moves every registered parameter to the GPU, so weights and input now match._

**Answer:** You moved the data but not the model. The input is on cuda:0 while the layer weights are still on the CPU, and a matmul needs both operands on the same device. Call model.to("cuda") (it moves every registered parameter at once) and x = x.to("cuda"). A clean pattern is to pick one device up front and send both the model and every batch to it.

</details>

**Problem 3.** You want to confirm a freshly built MLP Linear(10&rarr;32) &rarr; ReLU &rarr; Linear(32&rarr;2) has the number of trainable parameters you expect. How do you compute it by hand, and how do you check it in code?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply the per-linear-layer formula in*out + out to each nn.Linear. — _A linear layer holds a weight matrix (in*out numbers) plus one bias per output (out numbers)._
- First layer: 10*32 + 32 = 352. Second layer: 32*2 + 2 = 66. ReLU adds 0. — _ReLU is a parameter-free activation, so it contributes nothing to the count._
- Sum to 418, then verify with sum(p.numel() for p in model.parameters()). — _p.numel() counts the elements in each registered parameter tensor; summing over parameters() totals them._

**Answer:** Use in*out + out per linear layer. First layer: $10\times32+32 = 352$. ReLU: $0$. Second layer: $32\times2+2 = 66$. Total $= 418$. Check it with sum(p.numel() for p in model.parameters()), which iterates every registered weight tensor and sums their element counts &mdash; it should print 418.

</details>